# Run 1307 AI code from GCS in Colab

This notebook is meant to be opened from GitHub in Google Colab.
It syncs your `1307` code from a Google Cloud Storage bucket into `/content/1307`,
installs dependencies, checks GPU availability, and runs your entry script.

## 1) Configure

Fill in these values before running the rest of the notebook.

In [ ]:
GCP_PROJECT = ""  # e.g. "my-geothermal-project"
GCS_BUCKET = ""   # e.g. "gis-final-project"
GCS_PREFIX_1307 = "GIS Final Project/1307"

LOCAL_1307_DIR = "/content/1307"
REQUIREMENTS_FILE = "requirements.txt"  # relative to LOCAL_1307_DIR
EXTRA_PIP_PACKAGES = ""  # optional, space-separated

ENTRY_SCRIPT = "train.py"  # e.g. "main.py"
ENTRY_ARGS = ""            # e.g. "--config configs/train.yaml --epochs 50"

## 2) Authenticate and sync from GCS

In [ ]:
import shlex
import subprocess
from pathlib import Path

from google.colab import auth

def run(cmd, cwd=None):
    print("$", " ".join(shlex.quote(str(c)) for c in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not GCP_PROJECT or not GCS_BUCKET:
    raise ValueError("Set GCP_PROJECT and GCS_BUCKET in the config cell first.")

auth.authenticate_user()
run(["gcloud", "config", "set", "project", GCP_PROJECT])

local_dir = Path(LOCAL_1307_DIR)
local_dir.mkdir(parents=True, exist_ok=True)

_prefix = GCS_PREFIX_1307.strip("/")
src = f"gs://{GCS_BUCKET}/{_prefix}" if _prefix else f"gs://{GCS_BUCKET}"
run(["gsutil", "-m", "rsync", "-r", src, str(local_dir)])

print("Synced to", local_dir)

## 3) Inspect synced files

In [ ]:
from pathlib import Path

root = Path(LOCAL_1307_DIR)
if not root.exists():
    raise FileNotFoundError(f"Missing local code directory: {root}")

items = sorted(p.name for p in root.iterdir())
print(f"Top-level files/folders in {root}:")
for name in items[:200]:
    print(" -", name)

## 4) Install dependencies

In [ ]:
import sys
import shlex
from pathlib import Path

req_path = Path(LOCAL_1307_DIR) / REQUIREMENTS_FILE

run([sys.executable, "-m", "pip", "install", "-U", "pip"])
if req_path.exists():
    run([sys.executable, "-m", "pip", "install", "-r", str(req_path)])
else:
    print(f"No requirements file found at: {req_path}")

extra = EXTRA_PIP_PACKAGES.strip()
if extra:
    run([sys.executable, "-m", "pip", "install", *shlex.split(extra)])

## 5) Check GPU runtime

In [ ]:
import subprocess

try:
    run(["nvidia-smi"])
except subprocess.CalledProcessError:
    print("nvidia-smi failed. In Colab: Runtime -> Change runtime type -> GPU.")

try:
    import torch
    print("torch version:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
except Exception as e:
    print("Torch check skipped:", e)

## 6) Run your 1307 entry script

In [ ]:
import shlex
import sys
from pathlib import Path

entry = Path(LOCAL_1307_DIR) / ENTRY_SCRIPT
if not entry.exists():
    raise FileNotFoundError(
        f"ENTRY_SCRIPT not found: {entry}\n"
        "Update ENTRY_SCRIPT in the config cell."
    )

cmd = [sys.executable, str(entry)]
args = ENTRY_ARGS.strip()
if args:
    cmd.extend(shlex.split(args))

run(cmd, cwd=LOCAL_1307_DIR)